Genetic Algo with Roulette Wheel Selection, Single Point Crossover

In [46]:
import numpy as np
import random

# 1. Define the specific chromosomes given in the problem
population_data = [
    [12, 5, 23, 8],
    [2, 21, 18, 3],
    [10, 4, 13, 14],
    [20, 1, 10, 6],
    [1, 4, 13, 19],
    [20, 5, 17, 1]
]

# Convert to numpy array for easier handling
population = np.array(population_data)

# 2. Define the Fitness Function
def calculate_fitness(chromosome):
    # a, b, c, d corresponds to indices 0, 1, 2, 3
    a, b, c, d = chromosome
    val = (a + 2*b + 3*c + 4*d) - 30
    error = abs(val)
    # We add 1 to avoid division by zero. Higher is better.
    return 1.0 / (1.0 + error)

# 3. Roulette Wheel Selection
def roulette_wheel_selection(population, fitness_scores):
    total_fitness = sum(fitness_scores)
    probs = [f / total_fitness for f in fitness_scores]
    
    # Select indices based on probability
    # We select 2 parents
    indices = np.random.choice(len(population), size=2, p=probs, replace=True)
    return population[indices[0]], population[indices[1]]

# 4. Crossover (Single Point)
def crossover(parent1, parent2):
    if random.random() > 0.8: # 80% crossover rate
        point = random.randint(1, len(parent1) - 1)
        child1 = np.concatenate((parent1[:point], parent2[point:]))
        child2 = np.concatenate((parent2[:point], parent1[point:]))
        return child1, child2
    return parent1, parent2

# 5. Mutation
def mutate(child):
    if random.random() < 0.1: # 10% mutation rate
        # Pick a random gene and add a small random integer (-1 to 1)
        idx = random.randint(0, len(child) - 1)
        child[idx] = child[idx] + random.randint(-2, 2)
        # Ensure values don't become negative if that's a constraint (optional)
        child[idx] = max(0, child[idx]) 
    return child

# --- Main GA Loop ---
generations = 100  # Keeping it short for demonstration

print("--- Initial Population & Fitness ---")
for i, ind in enumerate(population):
    print(f"Ch{i+1}: {ind}, Fitness: {calculate_fitness(ind):.4f}, Value: {(ind[0]+2*ind[1]+3*ind[2]+4*ind[3])-30}")

for gen in range(generations):
    # Calculate fitness for all
    fitness_scores = [calculate_fitness(ind) for ind in population]
    
    new_population = []
    
    # Create next generation
    while len(new_population) < len(population):
        # Selection
        p1, p2 = roulette_wheel_selection(population, fitness_scores)
        
        # Crossover
        c1, c2 = crossover(p1, p2)
        
        # Mutation
        c1 = mutate(c1)
        c2 = mutate(c2)
        
        new_population.extend([c1, c2])
    
    population = np.array(new_population[:len(population_data)]) # Keep population size constant

    # Check for perfect solution
    best_fitness = max(fitness_scores)
    # print(f"Gen {gen+1}: Best Fitness: {best_fitness:.4f}")
    if best_fitness == 1.0:
        print(f"\nSolution Found in Gen {gen}!")
        break

# Final Results
best_idx = np.argmax([calculate_fitness(ind) for ind in population])
best_sol = population[best_idx]
val = (best_sol[0] + 2*best_sol[1] + 3*best_sol[2] + 4*best_sol[3]) - 30

print("\n--- Final Best Solution ---")
print(f"Chromosome: {best_sol}")
print(f"Function Value (Target 0): {val}")

--- Initial Population & Fitness ---
Ch1: [12  5 23  8], Fitness: 0.0106, Value: 93
Ch2: [ 2 21 18  3], Fitness: 0.0123, Value: 80
Ch3: [10  4 13 14], Fitness: 0.0119, Value: 83
Ch4: [20  1 10  6], Fitness: 0.0213, Value: 46
Ch5: [ 1  4 13 19], Fitness: 0.0105, Value: 94
Ch6: [20  5 17  1], Fitness: 0.0179, Value: 55

Solution Found in Gen 99!

--- Final Best Solution ---
Chromosome: [2 2 8 0]
Function Value (Target 0): 0


GA using pygad library for diabetes dataset 

In [ ]:
import pygad
import numpy as np
from sklearn.datasets import load_diabetes
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

# 1. Load Data (Diabetes Medical Dataset)
data = load_diabetes()
X = data.data
y = data.target
feature_names = data.feature_names

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# 2. Define Fitness Function for Feature Selection
def fitness_func_fs(ga_instance, solution, solution_idx):
    # 'solution' is a binary array ( [1, 0, 1, ...])
    # Select features where gene is 1
    selected_indices = [i for i, gene in enumerate(solution) if gene == 1]
    
    # If no features selected, return 0 fitness
    if len(selected_indices) == 0:
        return 0
        
    X_train_sel = X_train[:, selected_indices]
    X_test_sel = X_test[:, selected_indices]
    
    # Train Model
    model = LinearRegression()
    model.fit(X_train_sel, y_train)
    predictions = model.predict(X_test_sel)
    
    # Calculate Error
    mse = mean_squared_error(y_test, predictions)
    
    # Fitness = Inverse of MSE (Minimize Error -> Maximize Fitness)
    return 1.0 / (mse + 1e-5)

# 3. Configure PyGAD for Binary Optimization
ga_fs = pygad.GA(
    num_generations=20,
    num_parents_mating=4,
    fitness_func=fitness_func_fs,
    sol_per_pop=10,
    num_genes=len(feature_names), 
    gene_type=int,                
    init_range_low=0,
    init_range_high=2,            
    parent_selection_type="tournament", 
    crossover_type="single_point",
    mutation_type="random",
    mutation_probability=0.1
)

# 4. Run
print("\nRunning GA for Feature Selection...")
ga_fs.run()

# 5. Results
best_sol, best_fit, best_idx = ga_fs.best_solution()
selected_features = [feature_names[i] for i, gene in enumerate(best_sol) if gene == 1]

print(f"\n--- Part B Result ---")
print(f"Original Features: {len(feature_names)}")
print(f"Selected Features: {len(selected_features)}")
print(f"Feature List: {selected_features}")
print(f"Best MSE (Error): {1.0/best_fit:.2f}")


Running GA for Feature Selection...

--- Part B Result ---
Original Features: 10
Selected Features: 6
Feature List: ['sex', 'bmi', 's1', 's3', 's5', 's6']
Best MSE (Error): 2758.69


GA using NiaPy library for minimizing f(x) = (a + 2b + 3c + 4d)

In [100]:
import numpy as np
from niapy.problems import Problem
from niapy.task import Task
from niapy.algorithms.basic import GeneticAlgorithm

# 1. Define the Custom Problem Class
class EquationProblem(Problem):
    def __init__(self, dimension=4, lower=1, upper=30):
        # We have 4 variables (a, b, c, d)
        super().__init__(dimension=dimension, lower=lower, upper=upper)
        self.weights = np.array([1, 2, 3, 4])
        self.target = 30

    def _evaluate(self, x):
        # NiaPy passes 'x' as a numpy array of floats
        # We round them to integers for this specific integer-like problem
        params = np.round(x)
        
        # Calculate: (a + 2b + 3c + 4d)
        result = np.sum(params * self.weights)
        
        # We want to minimize the difference from 30
        error = abs(result - self.target)
        return error

# 2. Setup the Task
# We define the optimization task (Minimization is default)
task = Task(problem=EquationProblem(), max_evals=10000)

# 3. Configure and Run Genetic Algorithm
# NiaPy handles population initialization internally
algo = GeneticAlgorithm(population_size=100, crossover_probability=0.8, mutation_probability=0.2)
best_x, best_fit = algo.run(task)

# 4. Results
print("--- Part A: NiaPy Equation Solver ---")
print(f"Best Solution (a, b, c, d): {np.round(best_x)}")
print(f"Final Error: {best_fit} (Target 0.0)")

--- Part A: NiaPy Equation Solver ---
Best Solution (a, b, c, d): [3. 5. 3. 2.]
Final Error: 0.0 (Target 0.0)
